# Problem Set 5: Randomized Control Trials

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due **Monday 10/26** (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Analyzing randomized control trial data (Oregon Health Insurance Experiment).

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up

In this problem set we replicate parts of the **Oregon Health Insurance Experiment** — the famous RCT in which Oregon ran a lottery for Medicaid slots, discussed in class (Finkelstein et al. 2012). The data loads directly from a link.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

## Part 1: Explore the sample

The dataset has one row per lottery participant. Key variable groups:

- **Sample:** `treatment` (won the lottery), `sample_inp_resp` (responded to the in-person survey), `weight_total_inp` (survey weight)
- **Demographics:** `gender_inp`, `hispanic_inp`, `race_*_inp`, `age_*_inp`, `itvw_english_inp`
- **Pre-lottery health:** `*_dx_pre_lottery` (diagnoses before the lottery)
- **Outcomes:** `*_post_lottery`, `hasmedicaid`, `bmi_inp`, `phqtot_high` (depression screen), `any_oop_inp` / `catastrophic_exp_inp` (out-of-pocket / catastrophic spending)

In [ ]:
df = pd.read_stata("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/oregonhie.dta", convert_categoricals=False)
print(df.shape)
df.head()

In [ ]:
print(df.treatment.value_counts(), "\n")
print(pd.crosstab(df.treatment, df.gender_inp), "\n")
print(pd.crosstab(df.sample_inp_resp, df.treatment))

**Question 1.** How many people participated in the trial? How many are in the treatment and control groups?

**Answer:** *[your answer here]*

**Question 2.** Our analyses below use the *analytic sample* — people with `sample_inp_resp == 1`, who responded to the in-person survey. How many people are in this analytic sample, and how many of them are in treatment and control?

**Answer:** *[your answer here]*

## Part 2: The first stage — did winning the lottery mean getting Medicaid?

Winning the lottery gave people the *opportunity* to enroll in Medicaid — not everyone did. We regress `hasmedicaid` on `treatment` in the analytic sample, using the survey weights (`weights=`, the equivalent of Stata's `[pw=...]`) and robust standard errors.

In [ ]:
analytic = df[df.sample_inp_resp == 1].copy()

first_stage = smf.wls("hasmedicaid ~ treatment", analytic,
                      weights=analytic.weight_total_inp).fit(cov_type="HC1")
first_stage.summary().tables[1]

**Question 3.** How much did winning the lottery increase the probability of having Medicaid? Given that not everyone who won enrolled, how would you define the *compliers* in this experiment?

**Answer:** *[your answer here]*

## Part 3: The balance table

An RCT is only convincing if randomization actually "worked" — treatment and control should look similar on characteristics determined *before* the lottery. We regress each demographic and pre-lottery health variable on `treatment` (a loop, like Stata's `foreach`) and collect the coefficients.

In [ ]:
balance_vars = ["gender_inp", "hispanic_inp", "race_white_inp", "race_black_inp",
                "race_nwother_inp", "age_19_34_inp", "age_35_49_inp"]

balance_regs = []
for v in balance_vars:
    sub = analytic.dropna(subset=[v])
    r = smf.wls(f"{v} ~ treatment", sub, weights=sub.weight_total_inp).fit(cov_type="HC1")
    balance_regs.append(r)

print(summary_col(balance_regs, stars=True, model_names=balance_vars,
                  regressor_order=["treatment"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs):,}"}))

In [ ]:
# Same idea for the pre-lottery health variables
health_vars = ["age_50_64_inp", "itvw_english_inp", "ast_dx_pre_lottery", "dia_dx_pre_lottery",
               "hbp_dx_pre_lottery", "chl_dx_pre_lottery", "ami_dx_pre_lottery"]

health_regs = []
for v in health_vars:
    sub = analytic.dropna(subset=[v])
    r = smf.wls(f"{v} ~ treatment", sub, weights=sub.weight_total_inp).fit(cov_type="HC1")
    health_regs.append(r)

print(summary_col(health_regs, stars=True, model_names=health_vars,
                  regressor_order=["treatment"], drop_omitted=True,
                  info_dict={"N": lambda r: f"{int(r.nobs):,}"}))

**Question 4.** Look at the `treatment` coefficients in the balance tables. Are the treatment and control groups balanced? How do you tell, and why does this matter for interpreting the results below?

**Answer:** *[your answer here]*

## Part 4: Outcomes

Now the payoff: did winning the lottery (and the Medicaid coverage it brought) change health outcomes? We start with hypertension diagnosed *after* the lottery.

In [ ]:
out1 = smf.wls("hbp_dx_post_lottery ~ treatment", analytic,
               weights=analytic.weight_total_inp).fit(cov_type="HC1")
out1.summary().tables[1]

**Question 5.** Choose **at least two other outcome variables** (e.g. `dia_dx_post_lottery`, `dep_dx_post_lottery`, `phqtot_high`, `any_oop_inp`, `catastrophic_exp_inp`, `bmi_inp`) and estimate the effect of treatment on each, using the same technique. Put your code in the cell below.

In [ ]:
# Your outcome regressions here


**Question 6.** How do you interpret the findings of this RCT overall — what did Medicaid coverage change, and what did it not? Are there any shortcomings of the experiment or the analysis you would flag?

**Answer:** *[your answer here]*

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS05"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")